<a href="https://colab.research.google.com/github/alireza33zz/alireza33zz.github.io/blob/main/SGL_SDSU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# The Julia bootstrap block
# This should be run for the first time to install Julia kernel, and then refresh this page (e.g., Ctrl-R)
# so that colab will redirect to the installed Julia kernel
# and then doing your own work

# 1. install latest Julia using jill.py
#    tip: one can install specific Julia version using e.g., jill install 1.7
!pip install jill && jill install --upstream Official --confirm
# 2. install IJulia kernel
! julia -e 'using Pkg; Pkg.add("IJulia"); using IJulia; installkernel("Julia")'
# 3. hot-fix patch to strip the version suffix of the installed kernel so that this notebook kernelspec is version agnostic
!jupyter kernelspec install $(jupyter kernelspec list | grep julia | tr -s ' ' | cut -d' ' -f3) --replace --name julia



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.3/88.3 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for fire: filename=fire-0.5.0-py2.py3-none-any.whl size=116934 sha256=ebd21f0395e1d10b00641789c4dd25b170504b9b8c2d4b1c3b491756e7829109
  Stored in directory: /root/.cache/pip/wheels/90/d4/f7/9404e5db0116bd4d43e5666eaa3e70ab53723e1e3ea40c9a95
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=4386b95601e71ed701ce67007578876435a1041da6febee3b9b7b9c17171c459
  Stored in directory: /root/.cache/pip/wheels/8b/f1/7f/5c94f0a7a505ca1c81cd1d9208ae2064675d97582078e6c769
Successfully built fire wget
JILL - Julia Installer 4 Linux (MacOS, Windows and FreeBSD) -- Light

querying release information from https://julialang-s3.julialang.org/bin/versions.json
----- Download Julia -----
downloading Julia release for 1.9.3-l

In [1]:
####
import Pkg
Pkg.add("JuMP")
import Pkg
Pkg.add("GLPK")
using JuMP, GLPK

# Create a new model

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed DiffRules ──────────── v1.15.1
   Installed Bzip2_jll ──────────── v1.0.8+0
   Installed CodecBzip2 ─────────── v0.8.1
   Installed IrrationalConstants ── v0.2.2
   Installed DiffResults ────────── v1.1.0
   Installed StaticArraysCore ───── v1.4.2
   Installed NaNMath ────────────── v1.0.2
   Installed SpecialFunctions ───── v2.3.1
   Installed OrderedCollections ─── v1.6.2
   Installed BenchmarkTools ─────── v1.3.2
   Installed MutableArithmetics ─── v1.3.3
   Installed TranscodingStreams ─── v0.10.1
   Installed SnoopPrecompile ────── v1.0.3
   Installed DataStructures ─────── v0.18.15
   Installed OpenSpecFun_jll ────── v0.5.5+0
   Installed ForwardDiff ────────── v0.10.36
   Installed MacroTools ─────────── v0.5.11
   Installed LogExpFunctions ────── v0.3.26
   Installed CodecZlib ──────────── v0.7.3
   Installed CommonSubexpressions ─ v0.3.0
   Installed Compat ───────────────

In [2]:
# Create a new model
Total_demand1 = zeros(4)
wind_energy1 = zeros(4)
solar_energy1 = zeros(4)
stored_energy1 = zeros(4)
##
Total_demand2 = [0.6, 0.67, 0.9, 0.8]
wind_energy2 = [0.1, 0.4, 0.8, 0.4]
solar_energy2 = [0, 0.9, 0.9, 0]
stored_energy2 = [0.85, 0.85, 0.85, 0.85]
###
price_wind_energy = [1.5, 1.1, 0.75, 1.1]
price_solar_energy = [1000, 0.5, 0.5, 1000]
price_stored_energy = [1.1, 1.5, 0.5, 0.3]
for i in 1:4
 println("In the $(i)nd scenario:")
 println("Total demand in this time interval is equal to $(100*Total_demand2[i]) MW")
 println("Maximum available generation in this time interval is equal to $(40*wind_energy2[i] + 50*solar_energy2[i] + 10*stored_energy2[i]) MW")

if ( 40*wind_energy2[i] + 50*solar_energy2[i] + 10*stored_energy2[i] < 100*Total_demand2[i])
  a = 100*Total_demand2[i] - (40*wind_energy2[i] + 50*solar_energy2[i] + 10*stored_energy2[i])
  println("Low Generation")
  println("$a MW must be supplied by external source at 40 Dollar per MW")
   model = Model(GLPK.Optimizer)
# Decision Variables
@variable(model, 0 <= wind_energy <= 40*wind_energy2[i]) # Wind energy integrated
@variable(model, 0 <= solar_energy <= 50*solar_energy2[i]) # Solar energy integrated
@variable(model, 0 <= stored_energy <= 10*stored_energy2[i]) # Energy to store for later use
# Objective: Minimize cost
@objective(model, Min, 20*wind_energy*price_wind_energy[i] + 25*solar_energy*price_solar_energy[i] + 10*stored_energy*price_stored_energy[i])
# Constraints
@constraint(model, energy_balance, wind_energy + solar_energy + stored_energy == 100*Total_demand2[i]-a)
# Solve the problem
optimize!(model)
wind_energy1[i] = value.(wind_energy)
solar_energy1[i] = value.(solar_energy)
stored_energy1[i] = value.(stored_energy)

println("Wind Gen = $(wind_energy1[i]) MW at $(20*price_wind_energy[i]) Dollar per MW")
println("Solar Gen = $(solar_energy1[i]) MW at $(25*price_solar_energy[i]) Dollar per MW")
println("Stored Consumption = $(stored_energy1[i]) MW at $(10*price_stored_energy[i]) Dollar per MW")
println("Total cost =  $(wind_energy1[i]*20*price_wind_energy[i]+solar_energy1[i]*25*price_solar_energy[i]+stored_energy1[i]*10*price_stored_energy[i] + a*40) Dollar")

  else
  println("Normal operation")
  model = Model(GLPK.Optimizer)
# Decision Variables
@variable(model, 0 <= wind_energy <= 40*wind_energy2[i]) # Wind energy integrated
@variable(model, 0 <= solar_energy <= 50*solar_energy2[i]) # Solar energy integrated
@variable(model, 0 <= stored_energy <= 10*stored_energy2[i]) # Energy to store for later use
# Objective: Minimize cost
@objective(model, Min, 20*wind_energy*price_wind_energy[i] + 25*solar_energy*price_solar_energy[i] + 10*stored_energy*price_stored_energy[i])
# Constraints
@constraint(model, energy_balance, wind_energy + solar_energy + stored_energy == 100*Total_demand2[i])
# Solve the problem
optimize!(model)
wind_energy1[i] = value.(wind_energy)
solar_energy1[i] = value.(solar_energy)
stored_energy1[i] = value.(stored_energy)

println("Wind Gen = $(wind_energy1[i]) MW at $(20*price_wind_energy[i]) Dollar per MW")
println("Solar Gen = $(solar_energy1[i]) MW at $(25*price_solar_energy[i]) Dollar per MW")
println("Stored Consumption = $(stored_energy1[i]) MW at $(10*price_stored_energy[i]) Dollar per MW")
a = 100*Total_demand2[i] - (40*wind_energy2[i] + 50*solar_energy2[i] + 10*stored_energy2[i])
println("Total cost =  $(wind_energy1[i]*20*price_wind_energy[i]+solar_energy1[i]*25*price_solar_energy[i]+stored_energy1[i]*10*price_stored_energy[i]+a*40) Dollar")
end
end

In the 1nd scenario:
Total demand in this time interval is equal to 60.0 MW
Maximum available generation in this time interval is equal to 12.5 MW
Low Generation
47.5 MW must be supplied by external source at 40 Dollar per MW
Wind Gen = 4.0 MW at 30.0 Dollar per MW
Solar Gen = 0.0 MW at 25000.0 Dollar per MW
Stored Consumption = 8.5 MW at 11.0 Dollar per MW
Total cost =  2113.5 Dollar
In the 2nd scenario:
Total demand in this time interval is equal to 67.0 MW
Maximum available generation in this time interval is equal to 69.5 MW
Normal operation
Wind Gen = 13.5 MW at 22.0 Dollar per MW
Solar Gen = 45.0 MW at 12.5 Dollar per MW
Stored Consumption = 8.5 MW at 15.0 Dollar per MW
Total cost =  887.0 Dollar
In the 3nd scenario:
Total demand in this time interval is equal to 90.0 MW
Maximum available generation in this time interval is equal to 85.5 MW
Low Generation
4.5 MW must be supplied by external source at 40 Dollar per MW
Wind Gen = 32.0 MW at 15.0 Dollar per MW
Solar Gen = 45.0 MW at